## OpenRouter API

In [3]:
import os
import requests
from IPython.display import JSON
import json

#### Get API key

In [4]:
API_KEY = os.environ["OPENROUTER_API_KEY"]      # get API key stored locally
BASE_URL = "https://openrouter.ai/api/v1"

### 1. Ask question
List of all parameters here: https://openrouter.ai/docs/api_reference/parameters

**Anthropic model costs**

| Model | Input $/M | Output $/M |
| :--- | :--- | :--- |
| **Haiku 4.5** | 1.00 | 5.00 |
| **Sonnet 5** | 2.00 | 10.00 |
| **Sonnet 4.6** | 3.00 | 15.00 |
| **Opus 5.5** | 4.00 | 20.00 |
| **Fable 5.1** | 10.00 | 50.00 |

In [5]:
QUESTION = ("Name one Drosophila gene. Just the name.")

payload = {
    "model": "anthropic/claude-sonnet-5",  # claude-sonnet-5, claude-opus-5.5, claude-fable-5.1
    "temperature": 0,
    "messages": [
        {"role": "user", "content": QUESTION},
    ],
    "max_tokens": 1000,
}

response = requests.post(
    url=f"{BASE_URL}/chat/completions",
    headers={"Authorization": "Bearer " + API_KEY},
    json=payload,
    timeout=(10, 180),
)

#### View answer

In [6]:
body = response.json()
print(body["choices"][0]["message"]["content"])

**white**

This is one of the most famous Drosophila melanogaster genes, first described by Thomas Hunt Morgan in 1910. It affects eye color, causing a white eye phenotype when mutated, instead of the normal red eye color.


#### View entire response

In [7]:
JSON(body)

<IPython.core.display.JSON object>

In [6]:
print(json.dumps(body, indent=2))

{
  "id": "gen-1790549709-WzFv4P1z2ujzudedRKbw",
  "object": "chat.completion",
  "created": 1790549709,
  "model": "anthropic/claude-sonnet-5",
  "provider": "Claude Platform on AWS",
  "system_fingerprint": null,
  "service_tier": "default",
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "end_turn",
      "message": {
        "role": "assistant",
        "content": "**white**\n\n(The *white* gene in *Drosophila melanogaster* affects eye pigmentation and was one of the first genes discovered in the organism.)",
        "refusal": null,
        "reasoning": null
      }
    }
  ],
  "usage": {
    "prompt_tokens": 22,
    "completion_tokens": 52,
    "total_tokens": 74,
    "cost": 0.000564,
    "is_byok": false,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "cache_write_tokens": 0,
      "audio_tokens": 0,
      "video_tokens": 0
    },
    "cost_details": {
      "upstream_inference_cost": 0

In [8]:
print("text    :", body["choices"][0]["message"]["content"])
print("model   :", body["model"], "via", body["provider"])
print("finish  :", body["choices"][0]["finish_reason"])
print("tokens  :", body["usage"]["prompt_tokens"], "in,", body["usage"]["completion_tokens"], "out")
print("cost    :", body["usage"]["cost"])

text    : **white**

This is one of the most famous Drosophila melanogaster genes, first described by Thomas Hunt Morgan in 1910. It affects eye color, causing a white eye phenotype when mutated, instead of the normal red eye color.
model   : anthropic/claude-sonnet-5 via Claude Platform on AWS
finish  : stop
tokens  : 22 in, 80 out
cost    : 0.000844


**Temperature**

Temperature = 0 is not a guarantee of determinism, only a very strong tendency. It asks for the most likely next token every time, but batching and floating-point arithmetic can still change an occasional token.

Cannot have both temperature = 0 and reasoning. These models reject thinking with a fixed temperature. So turning on reasoning costs  reproducibility, which is a reason to leave it off for anything the evaluation measures.


| Temperature | Distinct answers from 3 calls using Sonnet 4.6 |
| :--- | :--- |
| **0** | 1 |
| **1** | 3 |

### 2. Ask question with some error checking

In [9]:
def call(payload):
    """Send the payload to OpenRouter and return the response body.
    Raises RuntimeError with a readable message if the request was refused."""
    try:
        response = requests.post(
            f"{BASE_URL}/chat/completions",
            headers={"Authorization": "Bearer " + API_KEY},
            json=payload,
            timeout=(10, 300),          # 10 s to connect, 300 s to answer
        )
    except requests.exceptions.Timeout:
        raise RuntimeError("no answer in time: the network or the provider is slow. Try again.")

    body = response.json()
    if "error" in body:                 # OpenRouter reports its own errors inside an HTTP 200
        error = body["error"]
        raise RuntimeError(f"OpenRouter refused [{error.get('code')}]: {error.get('message')}")
    if "choices" not in body:
        raise RuntimeError(f"unexpected response, keys were {list(body)}")
    if body["choices"][0]["finish_reason"] == "length":
        print("WARNING: the answer was cut off by max_tokens; raise it if you need more.")
    return body

In [10]:
QUESTION = ("Name one Drosophila gene. Just the name.")

payload = {
    "model": "anthropic/claude-opus-5.5",  # claude-sonnet-5, claude-opus-5.5, claude-fable-5.1
    "temperature": 0,
    "messages": [
        {"role": "user", "content": QUESTION},
    ],
    "max_tokens": 1000,
}

try:
    body = call(payload)
    print(body["choices"][0]["message"]["content"])
except RuntimeError as error:
    print(error)

white


Now a failed call

In [11]:
broken = {**payload, "model": "anthropic/claude-fable-5.1", "max_tokens": 120000}

try:
    call(broken)
except RuntimeError as error:
    print(error)

OpenRouter refused [402]: This request requires more credits, or fewer max_tokens. You requested up to 120000 tokens, but can only afford 45677. To increase, visit https://openrouter.ai/workspaces/default/keys/7daa7cf32c06ce06958eff9f9222a6cf7b327f0ace6633c490b9e167e39fa332 and adjust the key's total limit


## 3. Three types of inputs

user: holds the direct question or action item

assistant: usually represents the past responses in a conversation history

system: establishes the behavior, formatting rules, or tone

This sequence of cells demonstrates that the LMM is stateless, it has no memory of prior conversations. You have to provide the record of the conversation through the 'assistant'.

The 'system' role establishes behavior, the tone of the reply for example. 

In [12]:
QUESTION = ("Name one Drosophila gene. Just the name.")

payload = {
    "model": "anthropic/claude-opus-5.5",  # claude-sonnet-5, claude-opus-5.5, claude-fable-5.1
    "temperature": 0,
    "messages": [
        {"role": "user", "content": QUESTION},
    ],
    "max_tokens": 1000,
}

response = requests.post(
    url=f"{BASE_URL}/chat/completions",
    headers={"Authorization": "Bearer " + API_KEY},
    json=payload,
    timeout=(10, 180),
)

body = response.json()
print(body["choices"][0]["message"]["content"])

white


In [13]:
QUESTION = ("What did you just say?")

payload = {
    "model": "anthropic/claude-opus-5.5",  # claude-sonnet-5, claude-opus-5.5, claude-fable-5.1
    "temperature": 0,
    "messages": [
        {"role": "user", "content": QUESTION},
    ],
    "max_tokens": 1000,
}

response = requests.post(
    url=f"{BASE_URL}/chat/completions",
    headers={"Authorization": "Bearer " + API_KEY},
    json=payload,
    timeout=(10, 180),
)

body = response.json()
print(body["choices"][0]["message"]["content"])

I haven't said anything yet. Your message is the first one in our conversation, so there's no earlier reply from me.

If you were expecting a previous response, it may be from a different session, since I can't see past conversations. Feel free to paste it here or ask your question again, and I'll help.


In [14]:
QUESTION_1 = ("Name one Drosophila gene. Just the name.")
QUESTION_2 = ("What did you just say?")

payload = {
    "model": "anthropic/claude-opus-5.5",  # claude-sonnet-5, claude-opus-5.5, claude-fable-5.1
    "temperature": 0,
    "messages": [
        {"role": "user", "content": QUESTION_1},
        {"role": "assistant", "content": "white"},
        {"role": "user", "content": QUESTION_2},
    ],
    "max_tokens": 1000,
}

response = requests.post(
    url=f"{BASE_URL}/chat/completions",
    headers={"Authorization": "Bearer " + API_KEY},
    json=payload,
    timeout=(10, 180),
)

body = response.json()
print(body["choices"][0]["message"]["content"])

I said **"white,"** which is the name of a Drosophila gene. It's one of the most famous fruit fly genes: Thomas Hunt Morgan discovered a mutation in it in 1910 that turns the flies' normally red eyes white. That finding helped establish that genes are located on chromosomes, since *white* is on the X chromosome.


In [22]:
QUESTION_1 = ("Name one Drosophila gene. Just the name.")
QUESTION_2 = ("What did you just say?")

SYSTEM_1 = ("Answer all questions as though you are a pirate.")

payload = {
    "model": "anthropic/claude-opus-5.5",  # claude-sonnet-5, claude-opus-5.5, claude-fable-5.1
    "temperature": 0,
    "messages": [
        {"role": "system", "content": SYSTEM_1},
        {"role": "user", "content": QUESTION_1},
        {"role": "assistant", "content": "white"},
        {"role": "user", "content": QUESTION_2},
    ],
    "max_tokens": 1000,
}

response = requests.post(
    url=f"{BASE_URL}/chat/completions",
    headers={"Authorization": "Bearer " + API_KEY},
    json=payload,
    timeout=(10, 180),
)

body = response.json()
print(body["choices"][0]["message"]["content"])

Arr, I said *white*, matey! 'Tis the name o' a gene in the wee fruit fly, *Drosophila melanogaster*. And beggin' yer pardon, I forgot me pirate tongue in that last answer!

Old Cap'n Thomas Hunt Morgan spied a white-eyed male among his red-eyed crew back in 1910, and that be one o' the first mutant genes ever charted. The gene makes a wee pump that hauls pigment into the fly's eyes. Without it, their eyes be white as a ghost ship's sails!
